# Exploring the OpenCell CSVs

A tour of the tables in `opencell_data/`:

| File | What it is |
|---|---|
| `opencell-library-metadata.csv` | One row per tagged OpenCell target: tag design (gRNA, donor, primers) and genotyping results |
| `opencell-localization-annotations.csv` | Manual subcellular localization labels per target, graded 3 (strongest) to 1 |
| `opencell-protein-abundance.csv` | HEK293T RNA (TPM) and protein abundance (nM, copies/cell) for about 19k genes |
| `opencell-protein-interactions.csv` | IP-MS interactions (bait → prey) with p-value, enrichment and stoichiometries |

`tables/` holds identical copies of these four files plus a `*-readme.csv` for each that describes the columns.
`PXD024909` is a saved PRIDE web page (HTML), not a data table.

In [ ]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)

DATA_DIR = Path("opencell_data")

## 1. Inventory: every file in the folder

In [ ]:
def md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while b := f.read(chunk):
            h.update(b)
    return h.hexdigest()

inventory = pd.DataFrame([
    {
        "path": str(p.relative_to(DATA_DIR)),
        "size_kb": round(p.stat().st_size / 1024, 1),
        "suffix": p.suffix or "(none)",
        "md5": md5(p),
    }
    for p in sorted(DATA_DIR.rglob("*")) if p.is_file()
])
inventory["duplicate_of_another_file"] = inventory.duplicated("md5", keep=False)
inventory

## 2. Load the four main tables

The top-level CSVs and the copies in `tables/` are the same files (see the md5 column above), so we load the top-level ones.

In [ ]:
NAMES = ["library-metadata", "localization-annotations", "protein-abundance", "protein-interactions"]
dfs = {name: pd.read_csv(DATA_DIR / f"opencell-{name}.csv") for name in NAMES}

summary = pd.DataFrame([
    {
        "table": name,
        "rows": len(df),
        "cols": df.shape[1],
        "columns": ", ".join(df.columns),
        "pct_missing_cells": round(100 * df.isna().mean().mean(), 1),
    }
    for name, df in dfs.items()
])
summary

## 3. Column descriptions (from the readme CSVs)

In [ ]:
readmes = pd.concat(
    [pd.read_csv(DATA_DIR / "tables" / f"opencell-{name}-readme.csv").assign(table=name) for name in NAMES],
    ignore_index=True,
)[["table", "column_name", "column_description"]]

with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
    display(readmes)

In [ ]:
def profile(df):
    """Per-column dtype, missingness and number of unique values."""
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_missing": df.isna().sum(),
        "pct_missing": (100 * df.isna().mean()).round(1),
        "n_unique": df.nunique(),
        "example": df.apply(lambda c: c.dropna().iloc[0] if c.notna().any() else None),
    })

## 4. Library metadata
One row per OpenCell target cell line (endogenous mNeonGreen11 tag in HEK293T).

In [ ]:
lib = dfs["library-metadata"]
display(lib.head())
profile(lib)

In [ ]:
print("Tagged terminus:")
display(lib["tagged_terminus"].value_counts())
print("Insertion justification type:")
display(lib["insertion_justification_type"].value_counts())
lib[["fraction_wt_alleles", "fraction_mng11_integrated_alleles", "fraction_other_alleles"]].describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5), sharey=True)
for ax, col in zip(axes, ["fraction_wt_alleles", "fraction_mng11_integrated_alleles", "fraction_other_alleles"]):
    lib[col].plot.hist(bins=40, ax=ax)
    ax.set_title(col)
plt.tight_layout()

## 5. Localization annotations
Labels are `;`-separated. Grade 3 is the most confident/prominent localization, grade 1 the weakest.

In [ ]:
loc = dfs["localization-annotations"]
display(loc.head())
profile(loc)

In [ ]:
grade_cols = ["annotations_grade_3", "annotations_grade_2", "annotations_grade_1"]

long = (
    loc.melt(id_vars=["target_name", "ensg_id"], value_vars=grade_cols, var_name="grade", value_name="label")
       .dropna(subset=["label"])
       .assign(label=lambda d: d["label"].str.split(";"))
       .explode("label")
       .assign(label=lambda d: d["label"].str.strip(), grade=lambda d: d["grade"].str[-1].astype(int))
)

label_counts = long.pivot_table(index="label", columns="grade", values="target_name", aggfunc="count", fill_value=0)
label_counts = label_counts[[3, 2, 1]]
label_counts["total"] = label_counts.sum(axis=1)
label_counts.sort_values("total", ascending=False)

In [ ]:
ax = label_counts.sort_values("total")[[3, 2, 1]].plot.barh(stacked=True, figsize=(8, 9))
ax.set_xlabel("number of targets")
ax.set_title("Localization labels by grade")
plt.tight_layout()

n_grade3 = loc["annotations_grade_3"].fillna("").str.split(";").apply(lambda x: len([s for s in x if s]))
print("Number of grade-3 labels per target:")
n_grade3.value_counts().sort_index()

## 6. Protein abundance
Covers about 19k genes, far more than the ~1.3k OpenCell targets. Protein measurements are only present for a subset.

In [ ]:
ab = dfs["protein-abundance"]
display(ab.head())
profile(ab)

In [ ]:
num = ["hek_rna_tpm", "hek_protein_conc_nm", "hek_protein_copy_number"]
display(ab[num].describe())

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, col in zip(axes, num):
    np.log10(ab[col][(ab[col] > 0) & np.isfinite(ab[col])]).plot.hist(bins=60, ax=ax)
    ax.set_title(f"log10({col})")
plt.tight_layout()

In [ ]:
both = ab.dropna(subset=["hek_rna_tpm", "hek_protein_copy_number"])
both = both[(both.hek_rna_tpm > 0) & (both.hek_protein_copy_number > 0)]
plt.figure(figsize=(5, 5))
plt.scatter(np.log10(both.hek_rna_tpm), np.log10(both.hek_protein_copy_number), s=3, alpha=0.3)
plt.xlabel("log10 RNA TPM"); plt.ylabel("log10 protein copies/cell")
r = np.corrcoef(np.log10(both.hek_rna_tpm), np.log10(both.hek_protein_copy_number))[0, 1]
plt.title(f"RNA vs protein (n={len(both)}, Pearson r on logs = {r:.2f})")

## 7. Protein interactions
Each row is one bait (OpenCell target) → prey (interactor) edge from IP-MS. `pval` looks like -log10(p), given the values around 5. Note: 32 rows have `interaction_stoichiometry = inf`, so filter with `np.isfinite` before taking logs or means.

In [ ]:
ppi = dfs["protein-interactions"]
display(ppi.head())
profile(ppi)

In [ ]:
print("rows (directed edges):", len(ppi))
print("unique baits (targets):", ppi.target_gene_name.nunique())
print("unique preys (interactors):", ppi.interactor_gene_name.nunique())
print("unique genes overall:", len(set(ppi.target_gene_name) | set(ppi.interactor_gene_name)))

pairs = ppi[["target_ensg_id", "interactor_ensg_id"]].dropna()
undirected = {tuple(sorted(p)) for p in pairs.itertuples(index=False)}
print("unique undirected pairs:", len(undirected))
print("self-edges (bait == prey):", (ppi.target_ensg_id == ppi.interactor_ensg_id).sum())

ppi[["pval", "enrichment", "interaction_stoichiometry", "abundance_stoichiometry"]].describe()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 3.5))
ppi.groupby("target_gene_name").size().plot.hist(bins=50, ax=axes[0], title="preys per bait")
ppi.groupby("interactor_gene_name").size().plot.hist(bins=50, ax=axes[1], title="baits per prey", logy=True)
axes[2].scatter(ppi.enrichment, ppi.pval, s=2, alpha=0.2)
axes[2].set_xlabel("enrichment"); axes[2].set_ylabel("pval"); axes[2].set_title("volcano-style")
for col, c in [("interaction_stoichiometry", "C1"), ("abundance_stoichiometry", "C2")]:
    np.log10(ppi[col][(ppi[col] > 0) & np.isfinite(ppi[col])]).plot.hist(bins=60, ax=axes[3], alpha=0.5, label=col, color=c)
axes[3].legend(fontsize=7); axes[3].set_title("log10 stoichiometries")
plt.tight_layout()

print("Top 10 baits by number of preys:")
ppi.target_gene_name.value_counts().head(10)

## 8. How the tables connect (overlap on Ensembl gene ID)

In [ ]:
sets = {
    "library targets": set(lib.ensg_id.dropna()),
    "localization targets": set(loc.ensg_id.dropna()),
    "abundance genes": set(ab.ensg_id.dropna()),
    "PPI baits": set(ppi.target_ensg_id.dropna()),
    "PPI preys": set(ppi.interactor_ensg_id.dropna()),
}
overlap = pd.DataFrame(
    [[len(a & b) for b in sets.values()] for a in sets.values()],
    index=sets.keys(), columns=sets.keys(),
)
overlap